# TouchTrace — Sensor Model Training (Colab)

Train the Phase 2 IMU LSTM + MDN on fused CSD4CA data (touch + accelerometer + gyroscope).
The existing `touch.onnx` stays frozen; this notebook trains isolated Sensor candidates.

The model predicts **ΔIMU** (current − previous accel/gyro). Training uses a teacher-force warmup, then 4-step scheduled-sampling rollouts. Train/validation/test users come from the fixed `sensor_split.json` manifest. Every five epochs is retained for offline stochastic-rollout selection; deterministic AR MAE is only a severe-divergence guard. Section 9 compares independent and correlated innovations on validation users.

**Before you start:** Runtime → Change runtime type → **GPU** (T4).

Repo: [github.com/ginwzy/TouchTrace](https://github.com/ginwzy/TouchTrace)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repository

Training data `sensor/sensor_data.jsonl.gz` (~43 MB, 32,027 swipes) is already in the repo (touch points with interpolated accel/gyro, no magnetometer).

In [ ]:
from pathlib import Path

REPO = "TouchTrace"
REPO_URL = "https://github.com/ginwzy/TouchTrace.git"
ROOT = Path("/content") / REPO
TRAIN_DIR = ROOT / "train"
SENSOR_DIR = TRAIN_DIR / "sensor"

if not ROOT.exists():
    !git clone {REPO_URL}
else:
    !git -C {ROOT} pull --ff-only

assert SENSOR_DIR.is_dir(), f"Missing directory: {SENSOR_DIR}"
%cd {TRAIN_DIR}

data = SENSOR_DIR / "sensor_data.jsonl.gz"
assert data.is_file(), f"Missing {data} (cwd={Path.cwd()}). If this clone is behind, upload the gz in the last cell."
print(f"Data: {data} ({data.stat().st_size / 1e6:.1f} MB)")
!ls -lh sensor/sensor_data.jsonl.gz

## 3. Install dependencies

In [ ]:
!pip install -q tensorflow tensorflow-probability tf-keras tf2onnx onnxruntime matplotlib pytest

## 4. Verify TensorFlow sees the GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", gpus)

if not gpus:
    print("\n⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU, then rerun from the top.")
else:
    print("\n✓ GPU ready.")

## 5. Data check (human IMU baseline)

Seated `|a|` should sit near 9.81 (gravity). Walking variance should be higher. Gyro near 0 when seated.

Generation compare is skipped here because it needs the **new** ΔIMU `sensor.onnx` from this run. That happens in section 9.

In [ ]:
!python -m sensor.eval --skip-gen

## 6. Train

Defaults (from `sensor/config.py`): 13-d input (previous IMU + remaining-frame step + condition), 6-d MDN output (**ΔIMU**), separate z-scores for absolute IMU and deltas, 3px path subsample, 250 max epochs, batch 256, prepad on GPU. The fixed manifest keeps 40 train, 5 validation, and 5 final-test users disjoint. No magnetometer or geometric IMU rotation.

Three phases:

1. Teacher forcing (20 epochs). Save this once as the shared grouped warmup.
2. Branch from epoch 20 with `correction`, `natural`, or `hybrid`; the ramp reaches `p=1.0` over 50 epochs.
3. Continue to a fixed epoch budget, saving a rollout candidate every 5 epochs. Select on stochastic validation rollouts, then evaluate the chosen setup once on test users.

| Option | Default | Description |
|--------|---------|-------------|
| `EPOCHS` | `None` | Max epochs; `None` uses config (250) |
| `LITE` | `False` | Use 2×64 LSTM instead of 2×128 |
| `SEQUENCE` | `False` | Mac Metal fallback only; Colab/CUDA use prepad |
| `RUN_NAME` | `sensor-correction` | Isolated weights, norm, and checkpoint prefix |
| `SS_TARGET_MODE` | `correction` | One of `correction`, `natural`, `hybrid` |
| `INITIAL_WEIGHTS` | `None` | Shared epoch-20 warmup for an ablation branch |

In [ ]:
from sensor.config import model_config

assert model_config["ss_max"] == 1.0, "Need the ΔIMU training code (ss_max=1.0). Upload train/sensor/*.py or git pull."
assert int(model_config["ss_unroll_hops"]) >= 2, "Need closed-loop SS unroll (ss_unroll_hops>=2)."
assert float(model_config["ss_target_clip_z"]) > 0, "Need bounded SS correction targets."
assert bool(model_config["split_by_user"]), "Need user-grouped train/validation/test splits."
assert int(model_config["rollout_checkpoint_interval"]) == 5, "Need periodic rollout candidates."
assert float(model_config["ar_stability_max_mae_z"]) > 0, "Need the AR divergence guard."
print({key: model_config[key] for key in ("ss_max", "ss_temp", "ss_unroll_hops", "ss_target_clip_z", "split_by_user", "split_seed", "rollout_checkpoint_interval", "ar_stability_max_mae_z")})

EPOCHS = None  # None → config default (250)
LITE = False
SEQUENCE = False
RUN_NAME = "sensor-correction"
SS_TARGET_MODE = "correction"  # correction | natural | hybrid
INITIAL_WEIGHTS = None  # e.g. sensor/sensor-grouped-warmup.h5
INITIAL_EPOCH = 0       # set to 20 with the shared warmup

cmd = ["python", "-m", "sensor.train", "--run-name", RUN_NAME, "--ss-target-mode", SS_TARGET_MODE]
if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LITE:
    cmd.append("--lite")
if SEQUENCE:
    cmd.append("--sequence")
if INITIAL_WEIGHTS is not None:
    cmd.extend(["--initial-weights", INITIAL_WEIGHTS, "--initial-epoch", str(INITIAL_EPOCH)])

print("Running:", " ".join(cmd))
!{" ".join(cmd)}

## 7. (Optional) Run unit tests

In [ ]:
!python -m pytest -q

## 8. Export isolated ONNX diagnostic

In [ ]:
LITE = globals().get("LITE", False)
RUN_NAME = globals().get("RUN_NAME", "sensor-correction")
candidate_weights = f"sensor/{RUN_NAME}.h5"
candidate_norm = f"sensor/{RUN_NAME}_norm.json"
candidate_onnx = f"sensor/{RUN_NAME}.onnx"
export_cmd = (
    f"python -m sensor.convert --weights {candidate_weights} --norm {candidate_norm} "
    f"--output {candidate_onnx} --no-publish"
)
if LITE:
    export_cmd += " --lite"

!{export_cmd}
!ls -lh {candidate_weights} {candidate_norm} {candidate_onnx} sensor/{RUN_NAME}_candidate_e*.h5 2>/dev/null

## 9. Evaluate after export

Evaluate only the five validation users while tuning temperature/correlation. `rho=0` preserves independent MDN draws; `rho=0.65` is the retained historical-checkpoint diagnostic. Do not inspect the five test users until a target mode and checkpoint have been selected.

Use `--gen-limit 80` only for a quick collapse check. This cell inspects the branch's final weights; checkpoint selection must use `python -m sensor.sweep` over the periodic candidates with `--gen-limit 500 --seeds 42,43,44 --partition validation`. Only the selected setup is then evaluated with `--partition test`.

In [ ]:
!python -m sensor.eval --model {candidate_onnx} --norm {candidate_norm} --split-manifest sensor/sensor_split.json --partition validation --gen-limit 500 --temps 0.2,0.26,0.27,0.3,0.4 --rhos 0,0.65 --report

## 10. Preview plots

Same four figures as `python -m sensor.preview`: magnitudes by condition, seated channels, seated AR diagnostic, and validation histograms.

What to look at:
- `imu_diag_seated.png` — AR lines should move toward `tf-mean` / human, not stay flat
- `imu_hist.png` — walking `|a|` should not grow a long tail past ~12
- `imu_channels_seated.png` — gyro peaks should exist, not a flat line

In [ ]:
from IPython.display import Image, display
from pathlib import Path

!python -m sensor.preview --model {candidate_onnx} --norm {candidate_norm} --out-dir sensor/plots --limit 120

PLOTS = [
    "sensor/plots/imu_mags.png",
    "sensor/plots/imu_channels_seated.png",
    "sensor/plots/imu_diag_seated.png",
    "sensor/plots/imu_hist.png",
]
for path in PLOTS:
    print(path)
    if Path(path).is_file():
        display(Image(path, width=900))
    else:
        print("  missing")

## 11. Download weights and plots

In [ ]:
from google.colab import files
from pathlib import Path

LITE = globals().get("LITE", False)
RUN_NAME = globals().get("RUN_NAME", "sensor-correction")
downloads = [
    f"sensor/{RUN_NAME}.h5",
    f"sensor/{RUN_NAME}_last.h5",
    f"sensor/{RUN_NAME}_norm.json",
    f"sensor/{RUN_NAME}.onnx",
    "sensor/sensor_split.json",
]
downloads.extend(str(path) for path in sorted(Path("sensor").glob(f"{RUN_NAME}_candidate_e*.h5")))
downloads.extend(str(path) for path in sorted(Path("sensor").glob(f"{RUN_NAME}_ss_p*.h5")))
downloads.extend(globals().get("PLOTS", []))

for name in downloads:
    if Path(name).exists():
        print(f"Downloading {name} ...")
        files.download(name)
    else:
        print(f"Skip (not found): {name}")

---

### Using local code instead of GitHub

If these ΔIMU changes are not on GitHub yet, upload into `train/sensor/` (cwd after clone):

`config.py`, `features.py`, `split.py`, `sensor_split.json`, `train.py`, `generate.py`, `eval.py`, `sweep.py`, `preview.py`, `convert.py`

Then re-run from the install cell. The train cell checks grouped splits, periodic candidates, and the AR stability guard so an old clone fails loudly.

In [ ]:
# Uncomment to upload local files into the current directory:
# from google.colab import files
# uploaded = files.upload()
# print("Uploaded:", list(uploaded.keys()))